In [ ]:
# -*- coding: utf-8 -*-
"""
 INPUT: malicious_phish.csv (651k URLs with types)
 OUTPUT: phishing_detector_enhanced.pkl
"""

print("\n" + "="*80)
print(" PHISHGUARD ML PIPELINE - STARTING")
print("="*80 + "\n")

# SECTION 1: INSTALL PACKAGES

print(" Installing packages...")
import subprocess
import sys

packages = ['pandas', 'numpy', 'scikit-learn', 'xgboost', 'matplotlib', 'seaborn']
for package in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

print(" Packages installed!\n")

# SECTION 2: IMPORT LIBRARIES

print(" Importing libraries...")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from urllib.parse import urlparse, parse_qs
import re
import string
from datetime import datetime
import joblib
import warnings
import time
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve
)

warnings.filterwarnings('ignore')
print(" Libraries imported!\n")

# SECTION 3: FEATURE EXTRACTION CLASS

print("="*80)
print(" FEATURE EXTRACTOR SETUP")
print("="*80 + "\n")

class URLFeatureExtractor:
    """Extract 40+ features from URLs"""

    def __init__(self):
        self.suspicious_keywords = [
            'login', 'verify', 'account', 'confirm', 'update', 'secure',
            'banking', 'paypal', 'amazon', 'apple', 'microsoft', 'google',
            'password', 'signin', 'authenticate', 'authorize', 'validate',
            'credit', 'debit', 'card', 'bank', 'payment', 'transaction',
            'customer', 'support', 'urgent', 'suspended', 'limited', 'action'
        ]

        self.brand_names = [
            'amazon', 'paypal', 'apple', 'microsoft', 'google', 'facebook',
            'twitter', 'instagram', 'netflix', 'ebay', 'walmart', 'bank',
            'irs', 'ups', 'fedex', 'dhl', 'dropbox', 'icloud', 'linkedin'
        ]

        self.suspicious_tlds = ['tk', 'ml', 'ga', 'cf', 'ru', 'su', 'top', 'xyz']
        self.legitimate_tlds = ['com', 'org', 'net', 'edu', 'gov', 'co', 'uk']

    def extract_features(self, url):
        """Extract 40+ features from URL"""
        features = {}

        try:
            url = str(url).strip()
            if not url or url == 'nan' or len(url) < 3:
                return self._get_empty_features()

            # Add protocol if missing
            if not url.startswith('http'):
                url = 'http://' + url

            # === BASIC URL PROPERTIES ===
            features['url_length'] = len(url)
            features['has_https'] = 1 if url.startswith('https') else 0
            features['has_http'] = 1 if url.startswith('http') else 0

            # Parse URL
            parsed = urlparse(url)
            domain = parsed.netloc.lower()
            path = parsed.path
            query = parsed.query

            # === DOMAIN ANALYSIS ===
            features['domain_length'] = len(domain)
            features['domain_entropy'] = self._calculate_entropy(domain)
            features['domain_has_numbers'] = 1 if any(c.isdigit() for c in domain) else 0

            # === SUBDOMAIN ANALYSIS ===
            subdomains = domain.split('.')
            features['subdomain_count'] = len(subdomains) - 1
            features['excessive_subdomains'] = 1 if len(subdomains) > 4 else 0

            if len(subdomains) > 2:
                subdomain_lengths = [len(s) for s in subdomains[:-1]]
                features['avg_subdomain_length'] = np.mean(subdomain_lengths)
                features['subdomain_length_variance'] = np.var(subdomain_lengths)
            else:
                features['avg_subdomain_length'] = 0
                features['subdomain_length_variance'] = 0

            # === TLD ANALYSIS ===
            tld = subdomains[-1] if subdomains else ''
            features['tld_length'] = len(tld)
            features['suspicious_tld'] = 1 if tld in self.suspicious_tlds else 0
            features['legitimate_tld'] = 1 if tld in self.legitimate_tlds else 0
            features['country_tld'] = 1 if len(tld) == 2 else 0

            # === BRAND NAME SPOOFING ===
            domain_lower = domain.lower()
            brand_matches = [brand for brand in self.brand_names if brand in domain_lower]
            features['contains_brand_name'] = 1 if brand_matches else 0
            features['brand_name_count'] = len(brand_matches)

            max_similarity = 0
            for brand in self.brand_names:
                similarity = self._string_similarity(domain_lower, brand)
                max_similarity = max(max_similarity, similarity)
            features['max_brand_similarity'] = max_similarity

            # === PATH AND QUERY ===
            features['path_length'] = len(path)
            features['path_has_numbers'] = 1 if any(c.isdigit() for c in path) else 0
            features['query_length'] = len(query)
            features['query_param_count'] = len(parse_qs(query)) if query else 0
            features['has_suspicious_query'] = self._check_suspicious_query(query)

            # === SPECIAL CHARACTERS ===
            features['hyphen_count'] = url.count('-')
            features['underscore_count'] = url.count('_')
            features['dot_count'] = url.count('.')
            features['at_sign_count'] = url.count('@')
            features['percent_count'] = url.count('%')
            features['double_slash_count'] = url.count('//')

            # === CHARACTER DISTRIBUTION ===
            total_chars = len(domain)
            if total_chars > 0:
                features['digits_ratio'] = sum(1 for c in domain if c.isdigit()) / total_chars
                features['uppercase_ratio'] = sum(1 for c in domain if c.isupper()) / total_chars
                features['lowercase_ratio'] = sum(1 for c in domain if c.islower()) / total_chars
                features['special_chars_ratio'] = sum(1 for c in domain if c in '-._~') / total_chars
            else:
                features['digits_ratio'] = 0
                features['uppercase_ratio'] = 0
                features['lowercase_ratio'] = 0
                features['special_chars_ratio'] = 0

            # === SUSPICIOUS KEYWORDS ===
            url_lower = url.lower()
            keyword_matches = [kw for kw in self.suspicious_keywords if kw in url_lower]
            features['suspicious_keyword_count'] = len(keyword_matches)

            features['has_login'] = 1 if 'login' in url_lower else 0
            features['has_verify'] = 1 if 'verify' in url_lower else 0
            features['has_update'] = 1 if 'update' in url_lower else 0
            features['has_confirm'] = 1 if 'confirm' in url_lower else 0
            features['has_account'] = 1 if 'account' in url_lower else 0
            features['has_urgent'] = 1 if 'urgent' in url_lower else 0

            # === IP ADDRESS DETECTION ===
            features['has_ip_address'] = 1 if self._is_ip_address(domain) else 0
            features['is_invalid_ip'] = 1 if self._is_invalid_ip(domain) else 0

            # === URL ANOMALIES ===
            features['multiple_at_signs'] = 1 if url.count('@') > 1 else 0
            features['has_url_in_path'] = 1 if self._has_url_in_path(path) else 0
            features['has_port'] = 1 if ':' in domain else 0

            # === CHARACTER REPETITION ===
            features['max_char_repetition'] = self._max_char_repetition(domain)
            features['has_hyphen_in_domain'] = 1 if '-' in domain else 0

            # === URL ENCODING ===
            features['has_percent_encoding'] = 1 if '%' in url else 0
            features['percent_encoding_ratio'] = url.count('%') / len(url) if len(url) > 0 else 0

        except Exception as e:
            return self._get_empty_features()

        return features

    def _get_empty_features(self):
        """Return feature dict with all zeros"""
        return {
            'url_length': 0, 'has_https': 0, 'has_http': 0,
            'domain_length': 0, 'domain_entropy': 0, 'domain_has_numbers': 0,
            'subdomain_count': 0, 'excessive_subdomains': 0,
            'avg_subdomain_length': 0, 'subdomain_length_variance': 0,
            'tld_length': 0, 'suspicious_tld': 0, 'legitimate_tld': 0, 'country_tld': 0,
            'contains_brand_name': 0, 'brand_name_count': 0, 'max_brand_similarity': 0,
            'path_length': 0, 'path_has_numbers': 0, 'query_length': 0,
            'query_param_count': 0, 'has_suspicious_query': 0,
            'hyphen_count': 0, 'underscore_count': 0, 'dot_count': 0,
            'at_sign_count': 0, 'percent_count': 0, 'double_slash_count': 0,
            'digits_ratio': 0, 'uppercase_ratio': 0, 'lowercase_ratio': 0,
            'special_chars_ratio': 0, 'suspicious_keyword_count': 0,
            'has_login': 0, 'has_verify': 0, 'has_update': 0, 'has_confirm': 0,
            'has_account': 0, 'has_urgent': 0, 'has_ip_address': 0, 'is_invalid_ip': 0,
            'multiple_at_signs': 0, 'has_url_in_path': 0, 'has_port': 0,
            'max_char_repetition': 0, 'has_hyphen_in_domain': 0,
            'has_percent_encoding': 0, 'percent_encoding_ratio': 0
        }

    def _calculate_entropy(self, text):
        if not text or len(text) == 0:
            return 0
        entropy = 0
        text_len = len(text)
        for char in set(text):
            frequency = text.count(char) / text_len
            entropy -= frequency * np.log2(frequency)
        return entropy

    def _string_similarity(self, s1, s2):
        longer = s1 if len(s1) >= len(s2) else s2
        shorter = s2 if len(s1) >= len(s2) else s1
        if len(longer) == 0:
            return 1.0
        edit_distance = self._levenshtein_distance(longer, shorter)
        return (len(longer) - edit_distance) / float(len(longer))

    def _levenshtein_distance(self, s1, s2):
        if len(s1) < len(s2):
            return self._levenshtein_distance(s2, s1)
        if len(s2) == 0:
            return len(s1)
        previous_row = range(len(s2) + 1)
        for i, c1 in enumerate(s1):
            current_row = [i + 1]
            for j, c2 in enumerate(s2):
                insertions = previous_row[j + 1] + 1
                deletions = current_row[j] + 1
                substitutions = previous_row[j] + (c1 != c2)
                current_row.append(min(insertions, deletions, substitutions))
            previous_row = current_row
        return previous_row[-1]

    def _is_ip_address(self, domain):
        pattern = r'^(\d{1,3}\.){3}\d{1,3}$'
        if not re.match(pattern, domain):
            return False
        parts = domain.split('.')
        for part in parts:
            try:
                num = int(part)
                if num > 255:
                    return False
            except:
                return False
        return True

    def _is_invalid_ip(self, domain):
        parts = domain.split('.')
        if len(parts) != 4:
            return False
        for part in parts:
            if not part.isdigit():
                return False
            try:
                num = int(part)
                if num > 255:
                    return True
            except:
                return False
        return False

    def _has_url_in_path(self, path):
        return bool(re.search(r'https?://', path))

    def _max_char_repetition(self, text):
        if not text or len(text) == 0:
            return 0
        max_rep = 1
        current_rep = 1
        for i in range(1, len(text)):
            if text[i] == text[i-1]:
                current_rep += 1
                max_rep = max(max_rep, current_rep)
            else:
                current_rep = 1
        return max_rep

    def _check_suspicious_query(self, query):
        if not query:
            return 0
        suspicious_params = ['redirect', 'url', 'http', 'login', 'email', 'password']
        query_lower = query.lower()
        return 1 if any(param in query_lower for param in suspicious_params) else 0

print(" Feature Extractor ready!\n")

# SECTION 4: LOAD DATASET

print("="*80)
print(" LOADING DATASET")
print("="*80 + "\n")

try:
    df = pd.read_csv('/content/malicious_phish.csv')
    print(" Dataset loaded successfully!")
    print(f"  Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
    print(f"  Columns: {list(df.columns)}")
    print(f"\nFirst 5 rows:")
    print(df.head())
    print(f"\nLabel distribution:")
    print(df['type'].value_counts())
except Exception as e:
    print(f" Error loading dataset: {e}")
    import sys
    sys.exit(1)

# SECTION 5: PREPARE LABELS


print("\n" + "="*80)
print("  CONVERTING LABELS")
print("="*80 + "\n")

# Convert labels: phishing/defacement -> 1, benign -> 0
def convert_label(label):
    label_lower = str(label).lower().strip()
    if label_lower in ['phishing', 'defacement', 'malicious']:
        return 1
    elif label_lower in ['benign', 'legitimate']:
        return 0
    else:
        return 1  # Default to phishing if unclear

df['label'] = df['type'].apply(convert_label)

print("Label Conversion:")
for orig_type in df['type'].unique():
    new_label = convert_label(orig_type)
    count = (df['type'] == orig_type).sum()
    label_name = "Phishing" if new_label == 1 else "Legitimate"
    print(f"  {orig_type:15s} → {label_name:15s} ({count:,} URLs)")

print(f"\nFinal Label Distribution:")
print(f"  Legitimate (0): {(df['label']==0).sum():,}")
print(f"  Phishing (1):   {(df['label']==1).sum():,}")

# SECTION 6: EXTRACT FEATURES

print("\n" + "="*80)
print("  EXTRACTING 40+ FEATURES FROM {0:,} URLs".format(len(df)))
print("="*80 + "\n")

extractor = URLFeatureExtractor()

print(f"This will take 10-20 minutes (depending on your dataset size)...\n")

features_list = []
errors = 0
start_time = time.time()

for idx, row in df.iterrows():
    if idx % 50000 == 0 and idx > 0:
        elapsed = time.time() - start_time
        rate = idx / elapsed
        eta = (len(df) - idx) / rate if rate > 0 else 0
        pct = idx / len(df) * 100
        print(f"  Progress: {idx:,}/{len(df):,} ({pct:.1f}%) | "
              f"Rate: {rate:.0f} URLs/sec | ETA: {eta/60:.1f} min")

    try:
        url = str(row['url']).strip()
        label = row['label']

        if not url or url == 'nan' or len(url) < 3:
            errors += 1
            continue

        features = extractor.extract_features(url)
        features['url'] = url
        features['label'] = label
        features_list.append(features)

    except Exception as e:
        errors += 1
        continue

elapsed = time.time() - start_time

print(f"\n Feature extraction complete in {elapsed/60:.1f} minutes!")
print(f"  URLs processed: {len(features_list):,}")
print(f"  Errors: {errors:,}")

features_df = pd.DataFrame(features_list)
print(f"\n Features DataFrame created:")
print(f"   Shape: {features_df.shape}")
print(f"   Features per URL: {features_df.shape[1] - 2}")

# Clean NaN
nan_before = features_df.isnull().sum().sum()
features_df = features_df.dropna()

if nan_before > 0:
    print(f"   Cleaned NaN values: {nan_before}")

print(f"   Final shape: {features_df.shape}")

# SECTION 7: DATA ANALYSIS

print("\n" + "="*80)
print(" DATA ANALYSIS")
print("="*80 + "\n")

print("Class Distribution:")
for label_val in sorted(features_df['label'].unique()):
    count = (features_df['label'] == label_val).sum()
    pct = count / len(features_df) * 100
    bar = "█" * int(pct / 2)
    label_name = "Phishing" if label_val == 1 else "Legitimate"
    print(f"  {label_name:15s}: {count:8,} ({pct:5.1f}%) {bar}")

# SECTION 8: PREPARE DATA

print("\n" + "="*80)
print(" PREPARING DATA FOR MODELS")
print("="*80 + "\n")

X = features_df.drop(['url', 'label'], axis=1, errors='ignore')
y = features_df['label']

print(f"Features matrix: {X.shape}")

# Split: 70% train, 15% val, 15% test
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.15/(1-0.15), random_state=42, stratify=y_temp
)

print(f"  Train: {len(X_train):8,} ({len(X_train)/len(X)*100:5.1f}%)")
print(f"  Val:   {len(X_val):8,} ({len(X_val)/len(X)*100:5.1f}%)")
print(f"  Test:  {len(X_test):8,} ({len(X_test)/len(X)*100:5.1f}%)")

# Scale
print(f"\nScaling features...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)
print(f" Scaled!")

# SECTION 9: TRAIN MODELS

print("\n" + "="*80)
print(" TRAINING MODELS")
print("="*80 + "\n")

models_info = {}

# Logistic Regression
print("Training: Logistic Regression...")
start = time.time()
lr = LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1)
lr.fit(X_train_scaled, y_train)
lr_pred = lr.predict(X_val_scaled)
lr_acc = accuracy_score(y_val, lr_pred)
print(f"   Accuracy: {lr_acc:.4f}\n")
models_info['LogisticRegression'] = {'model': lr, 'accuracy': lr_acc}

# Random Forest
print("Training: Random Forest...")
start = time.time()
rf = RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
rf.fit(X_train_scaled, y_train)
rf_pred = rf.predict(X_val_scaled)
rf_acc = accuracy_score(y_val, rf_pred)
print(f"   Accuracy: {rf_acc:.4f}\n")
models_info['RandomForest'] = {'model': rf, 'accuracy': rf_acc}

# Gradient Boosting
print("Training: Gradient Boosting...")
gb = GradientBoostingClassifier(n_estimators=100, max_depth=7, learning_rate=0.1, random_state=42)
gb.fit(X_train_scaled, y_train)
gb_pred = gb.predict(X_val_scaled)
gb_acc = accuracy_score(y_val, gb_pred)
print(f"   Accuracy: {gb_acc:.4f}\n")
models_info['GradientBoosting'] = {'model': gb, 'accuracy': gb_acc}

# XGBoost
print("Training: XGBoost...")
xgb = XGBClassifier(n_estimators=100, max_depth=7, learning_rate=0.1, random_state=42, eval_metric='logloss')
xgb.fit(X_train_scaled, y_train)
xgb_pred = xgb.predict(X_val_scaled)
xgb_acc = accuracy_score(y_val, xgb_pred)
print(f"   Accuracy: {xgb_acc:.4f}\n")
models_info['XGBoost'] = {'model': xgb, 'accuracy': xgb_acc}

# Select best
best_name = max(models_info.items(), key=lambda x: x[1]['accuracy'])[0]
best_model = models_info[best_name]['model']
best_acc = models_info[best_name]['accuracy']

print("="*80)
print(f" BEST MODEL: {best_name}")
print(f"   Validation Accuracy: {best_acc:.4f}")
print("="*80 + "\n")

# SECTION 10: EVALUATE ON TEST SET

print("="*80)
print(" FINAL EVALUATION ON TEST SET")
print("="*80 + "\n")

y_pred = best_model.predict(X_test_scaled)
y_pred_proba = best_model.predict_proba(X_test_scaled)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
roc_auc = roc_auc_score(y_test, y_pred_proba)

print("PERFORMANCE METRICS:")
print(f"  Accuracy:  {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"  Precision: {precision:.4f} ({precision*100:.2f}%)")
print(f"  Recall:    {recall:.4f} ({recall*100:.2f}%)")
print(f"  F1-Score:  {f1:.4f}")
print(f"  ROC-AUC:   {roc_auc:.4f}")

cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

print(f"\nCONFUSION MATRIX:")
print(f"  True Negatives:  {tn:6,}  |  False Positives: {fp:6,}")
print(f"  False Negatives: {fn:6,}  |  True Positives:  {tp:6,}")

print(f"\nDETAILED REPORT:")
print(classification_report(y_test, y_pred, target_names=['Legitimate', 'Phishing'], zero_division=0))

# SECTION 11: VISUALIZATIONS

print("\nGenerating visualizations...")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
cm_plot = confusion_matrix(y_test, y_pred)
sns.heatmap(cm_plot, annot=True, fmt='d', cmap='Blues', ax=axes[0], cbar=False,
            xticklabels=['Legitimate', 'Phishing'], yticklabels=['Legitimate', 'Phishing'])
axes[0].set_title(f'Confusion Matrix - {best_name}', fontsize=12, fontweight='bold')
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
axes[1].plot(fpr, tpr, label=f'ROC curve (AUC = {roc_auc:.4f})', linewidth=2.5)
axes[1].plot([0, 1], [0, 1], 'k--', label='Random', linewidth=1.5)
axes[1].fill_between(fpr, tpr, alpha=0.2)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve', fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('model_evaluation.png', dpi=100, bbox_inches='tight')
print(" Saved: model_evaluation.png")
plt.close()

# SECTION 12: SAVE MODEL

print("\n" + "="*80)
print(" SAVING MODEL PACKAGE")
print("="*80 + "\n")

model_package = {
    'model': best_model,
    'model_name': best_name,
    'scaler': scaler,
    'feature_names': list(X.columns),
    'feature_count': len(X.columns),
    'accuracy': accuracy,
    'precision': precision,
    'recall': recall,
    'f1_score': f1,
    'roc_auc': roc_auc,
    'training_date': datetime.now().isoformat(),
    'dataset_size': len(features_df),
    'train_size': len(X_train),
    'val_size': len(X_val),
    'test_size': len(X_test),
    'feature_extractor_class': URLFeatureExtractor,
    'version': '1.0',
    'description': 'PhishGuard ML Model - Trained on malicious_phish.csv',
    'confusion_matrix': {
        'true_negatives': int(tn),
        'false_positives': int(fp),
        'false_negatives': int(fn),
        'true_positives': int(tp)
    }
}

filename = 'phishing_detector_enhanced.pkl'
joblib.dump(model_package, filename)

file_size = os.path.getsize(filename) / (1024 * 1024)

print(f"   Model saved: {filename}")
print(f"  File size: {file_size:.2f} MB")
print(f"\nPackage contents:")
print(f"   Model: {best_name}")
print(f"   Scaler: StandardScaler (fitted)")
print(f"   Features: {len(model_package['feature_names'])} features")
print(f"   Feature Extractor: URLFeatureExtractor class")


# SECTION 13: SUMMARY

print("\n" + "="*80)
print(" COMPLETE!")
print("="*80)

print(f"""

  INPUT:
   • Dataset: malicious_phish.csv
   • URLs: {len(features_df):,}
   • Classes: Legitimate & Phishing

  PROCESSING:
   • Features extracted: {len(X.columns)}
   • Train/Val/Test: {len(X_train):,} / {len(X_val):,} / {len(X_test):,}
   • Models trained: 4

  OUTPUT:
    Best Model: {best_name}
    Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)
    Precision: {precision:.4f}
    Recall: {recall:.4f}
    F1-Score: {f1:.4f}
    ROC-AUC: {roc_auc:.4f}
""")

print("\n  Download 'phishing_detector_enhanced.pkl' \n")


 PHISHGUARD ML PIPELINE - STARTING

 Installing packages...
 Packages installed!

 Importing libraries...
 Libraries imported!

 FEATURE EXTRACTOR SETUP

 Feature Extractor ready!

 LOADING DATASET

 Dataset loaded successfully!
  Shape: 651,191 rows × 2 columns
  Columns: ['url', 'type']

First 5 rows:
                                                 url        type
0                                   br-icloud.com.br    phishing
1                mp3raid.com/music/krizz_kaliko.html      benign
2                    bopsecrets.org/rexroth/cr/1.htm      benign
3  http://www.garage-pirenne.be/index.php?option=...  defacement
4  http://adventure-nicaragua.net/index.php?optio...  defacement

Label distribution:
type
benign        428103
defacement     96457
phishing       94111
malware        32520
Name: count, dtype: int64

  CONVERTING LABELS

Label Conversion:
  phishing        → Phishing        (94,111 URLs)
  benign          → Legitimate      (428,103 URLs)
  defacement      → Phishin